## Attribution

This notebook uses the `QumVQD` helper module (`QumVQD_Helpers.py`),
which contains code derived from the
[qumode_est_paper](https://github.com/rishabdchem/qumode_est_paper)
repository by Rishab Dutta, licensed under the MIT License.
See `NOTICE.md` and `third_party/LICENSE_qumode_est_paper.txt` in the
repository root for full attribution details.

# Run Parameters

In [ ]:
molecule = "H2" # Options: 'H2', 'H4'
hhdis = .5 # in Angstrom
SAVE_DIR = 'electronic_structure_output_H2' # Add a directory here to save VQD parameters and energies for each iteration. If SAVE_FILES = False, will not save.
PREV_STATE_PARAMS_LOCATION = None # Add a directory here to load VQD parameters from a previous run. If None, will not load and will start with ground state.
SAVE_FILES = True

#VQD Run Parameters
num_states = 8 # Number of states to compute.
ndepth = 4 # Number of layers in the qumode circuit ansatz.
beta_factor = 5.0 # The weight for the orthogonalization penalty in the VQD cost function.
niter = 3000 # Maximum number of iterations for the optimization.
threshold = 1e-7 # Convergence threshold for the optimization.
parallel_iterations = 3 # Number of parallel iterations to run in TensorFlow.
nfocks = [6] # Designates the number of modes and number of fock states for each mode. For example, [16] means 1 mode with 16 fock states, and [16, 16] means 2 modes with 16 fock states each.

# Prerequisite

In [33]:
import numpy as np
import time
import matplotlib.pyplot as plt
import json

In [34]:
import qutip as qt
import tensorflow as tf
import tensorflow_probability as tfp

In [35]:
from silence_tensorflow import silence_tensorflow
silence_tensorflow()

In [36]:
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

TensorFlow version: 2.21.0
GPU available: []


# Multi-Qumode VQD


In [37]:
from helpers.QumVQD_Helpers import *

## Get Operators

### Import operator mapping dictionary

In [38]:
fname = 'snap-four-qubit-dict_nd16.npy'

from pathlib import Path
data_path = Path.cwd().parent / 'data' / fname

op_dict = np.load(data_path, allow_pickle=True).item()

### H4

In [ ]:
if molecule == "H4":
  ham_h4 = ham_linear_sym_h4(hhdis)
  quditvec = [4, 4]
  nqubits = 8
  H_ops_h4, coeffs_h4 = get_multi_qumode_ops(ham_h4, op_dict, nqubits, quditvec, verb=1)
  H_ops_tf_h4 = prepare_ops_tf_no_hadamard(H_ops_h4)
  coeffs_tf_h4 = tf.constant(np.real(coeffs_h4), dtype=tf.float64)
  H4_full_bosonic = construct_bosonic_hamiltonian(H_ops_h4, coeffs_h4)
  H4_sliced = slice_bosonic_hamiltonian(H4_full_bosonic, n_electrons=4)
  energies, _ = np.linalg.eigh(H4_full_bosonic)
  fci_eigenenergies = compute_fci_states_h4(hhdis, 1000)

### H2

In [ ]:
if molecule == "H2":
  ham_h2 = ham_linear_sym_h2(hhdis)
  quditvec = [4]
  nqubits = 4
  H_ops, coeffs = get_multi_qumode_ops(ham_h2, op_dict, nqubits, quditvec)
  H_ops_tf = prepare_ops_tf_no_hadamard(H_ops)
  coeffs_tf = tf.constant(np.real(coeffs), dtype=tf.complex128)
  H2_full_bosonic = construct_bosonic_hamiltonian(H_ops_tf, coeffs_tf)
  H2_sliced = slice_bosonic_hamiltonian(H2_full_bosonic, n_electrons=2)
  energies, _ = np.linalg.eigh(H2_full_bosonic)
  fci_eigenenergies = compute_fci_states_h2(hhdis, 1000)
  print("FCI energies:", fci_eigenenergies)

Hartree-Fock energy for H2_sto-3g_singlet (2 electrons) is -1.042996274540095.
len(coeffs) = 15
building op 0
building ops for qumode 0
building op 5
building ops for qumode 0
building op 10
building ops for qumode 0


## run multi-qumode VQD

In [41]:
if PREV_STATE_PARAMS_LOCATION is not None:
  prev_state_params = [np.array(p) for p in np.load(PREV_STATE_PARAMS_LOCATION, allow_pickle=True)]
else:
  prev_state_params = []

In [ ]:
H_ops = None # If you want to simulate QumVQD at the gate level, set this to H_ops_tf_h4 for H4 or H_ops_tf for H2.
coeffs = None # If you want to simulate QumVQD at the gate level, set this to coeffs_tf_h4 for H4 or coeffs_tf for H2.

full_bosonic_H = H2_sliced # If you want to run QumVQD with the full Hamiltonian used to calculate expectaion values, set this to H4_full_bosonic for H4 or H2_full_bosonic for H2. If you want sliced versions, set thit to the output of slice_bosonic_hamiltonian()

energies, state_params, results, psis = find_excited_states_multi_qumode(
    H_ops=H_ops, coeffs=coeffs,
    nfocks=nfocks, ndepth=ndepth, num_states=num_states,
    beta_factor=beta_factor,
    niter=niter, threshold=threshold,
    full_bosonic_H = full_bosonic_H,
    prev_state_params = prev_state_params,
    save_files = SAVE_FILES,
    save_dir = SAVE_DIR,
    hhdis = hhdis,
)

print("Energies found:", energies)

num_prev_states = 0

Finding state 0...
  [saved checkpoint to electronic_structure_output after state 0]
State 0 energy: 0.263537
Converged: False
Energies found: [np.float64(0.26353656003384085)]
